# Notebook 16 — Temporal Validation
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Test whether Model D generalises across time by training on older games and
testing on newer ones, instead of the random stratified split used everywhere else in the
project. This answers Reviewer 2's comment: *"Limited generalizability — single dataset, single
platform, no temporal validation"* and the temporal half of Reviewer 1's Major Comment 1.

**This notebook DOES need the raw dataset.** `release_date` is marked `DROP` in NB01's column
decision matrix ("Post-concept metadata. Not used in any tier.") and is not present in
`games_features_t4.csv`. We reconstruct it from `data/raw/Dataset-2.csv`, apply the same
verification safety check used in NB15 (recompute the label from `pct_pos_total` and confirm it
matches the existing processed file row-for-row) before trusting that `release_date` is correctly
aligned to the existing T4 feature rows.

**This notebook is READ-ONLY with respect to existing files.** It loads the raw CSV and
`games_features_t4.csv`, but writes nothing back to either. The temporally-retrained model is
only kept in memory for evaluation, not saved over `model_d.pkl`.

**Inputs (read-only):**
- `data/raw/Dataset-2.csv`
- `data/processed/games_features_t4.csv`

**What stays fixed:** CatBoost hyperparameters (depth=8, learning_rate=0.05, iterations=1000,
l2_leaf_reg=3, scale_pos_weight=0.5 — Model D's actual config from NB04) and the T4 structured
features. Only the train/test split logic changes, from random stratified to chronological.

---
## Contents
1. Setup and Imports
2. Load Raw Dataset and Reconstruct the Filter
3. Verification — Confirm Row Alignment (reused pattern from NB15)
4. Parse Release Dates and Build the Temporal Split
5. Compare Class Balance: Older vs. Newer Games
6. Retrain Model D on the Temporal Split
7. Results Comparison: Random Split vs. Temporal Split
8. Summary


## 1. Setup and Imports

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, RANDOM_STATE, FEATURES_T4_CSV, TIER_FEATURES,
    RAW_CSV, COL_LABEL, COL_FILTER, CATBOOST_PARAMS,
)

CB_PARAMS = CATBOOST_PARAMS

print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {RANDOM_STATE}')


Project root : C:\Users\3214h\Documents\fyp-steam-reception
Random state : 42


## 2. Load Raw Dataset and Reconstruct the Filter

Identical filtering logic to NB15: remove sentinel `-1` rows, keep only games with
`num_reviews_total >= 100`. Should reproduce the same 20,383-game working set.


In [2]:
assert RAW_CSV.exists(), f'Not found: {RAW_CSV} — check the path matches your project layout.'

df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Loaded raw: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

assert 'release_date' in df_raw.columns, 'release_date column not found — check raw CSV schema.'
assert COL_LABEL in df_raw.columns
assert COL_FILTER in df_raw.columns

df_filtered = df_raw[
    (df_raw[COL_LABEL] != -1) &
    (df_raw[COL_FILTER] != -1) &
    (df_raw[COL_FILTER] >= 100)
].reset_index(drop=True)

print(f'After filtering: {df_filtered.shape[0]:,} rows')
print(f'Expected (per NB01): 20,383 rows')


Loaded raw: 89,618 rows x 47 columns
After filtering: 20,383 rows
Expected (per NB01): 20,383 rows


## 3. Verification — Confirm Row Alignment

Same safety check as NB15 Section 3: reconstruct the 75% threshold label from `pct_pos_total`
and confirm it matches `games_features_t4.csv`'s existing label row-for-row. If this passes, we
can trust that `release_date` (extracted in the same row order) is correctly aligned to the T4
feature rows too. **Do not proceed past this cell if verification fails.**


In [3]:
df_filtered['reception_label_75_reconstructed'] = (
    df_filtered[COL_LABEL] >= 75
).astype(int)

assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV}'
df_t4 = pd.read_csv(FEATURES_T4_CSV)

VERIFICATION_PASSED = False
if df_t4.shape[0] != df_filtered.shape[0]:
    print(f'*** WARNING: row count mismatch — games_features_t4.csv has {df_t4.shape[0]:,} rows, '
          f'reconstructed filter has {df_filtered.shape[0]:,} rows. DO NOT PROCEED.')
else:
    matches = (df_filtered['reception_label_75_reconstructed'].values == df_t4[TARGET_COL].values)
    match_rate = matches.mean() * 100
    print(f'Row-for-row label agreement: {match_rate:.2f}%')
    if match_rate > 99.5:
        print('VERIFIED — safe to proceed with release_date alignment.')
        VERIFICATION_PASSED = True
    else:
        print('*** WARNING: row order does not match. DO NOT proceed to Section 4.')
        print('*** (See NB15 Section 3 for the same check and troubleshooting notes.)')


Row-for-row label agreement: 100.00%
VERIFIED — safe to proceed with release_date alignment.


## 4. Parse Release Dates and Build the Temporal Split

Games are sorted by `release_date` ascending. The earliest ~80% (by count, matching the original
80/20 split ratio for fair comparison) become the training set; the most recent ~20% become the
test set. Any rows with an unparseable date are reported and excluded.


In [4]:
assert VERIFICATION_PASSED, 'Verification failed in Section 3 — resolve before continuing.'

df_filtered['release_date_parsed'] = pd.to_datetime(df_filtered['release_date'], errors='coerce')

n_unparseable = df_filtered['release_date_parsed'].isna().sum()
print(f'Unparseable release dates: {n_unparseable} ({n_unparseable/len(df_filtered)*100:.2f}%)')

valid_mask = df_filtered['release_date_parsed'].notna()
print(f'Games with a valid date: {valid_mask.sum():,} of {len(df_filtered):,}')

# Uses config.py's TIER_FEATURES[4] directly — verified via verify_tier_order.py.
feature_cols = TIER_FEATURES[4]
missing = [c for c in feature_cols if c not in df_t4.columns]
assert not missing, f'Columns missing from CSV: {missing}'
X_all = df_t4[feature_cols].values
y_all = df_t4[TARGET_COL].values

X_valid = X_all[valid_mask.values]
y_valid = y_all[valid_mask.values]
dates_valid = df_filtered.loc[valid_mask, 'release_date_parsed'].values

# Sort by date ascending, then split 80/20 by position (chronological, not random)
sort_idx = np.argsort(dates_valid)
X_sorted = X_valid[sort_idx]
y_sorted = y_valid[sort_idx]
dates_sorted = dates_valid[sort_idx]

split_point = int(len(X_sorted) * 0.8)
X_train_temporal, X_test_temporal = X_sorted[:split_point], X_sorted[split_point:]
y_train_temporal, y_test_temporal = y_sorted[:split_point], y_sorted[split_point:]
dates_train, dates_test = dates_sorted[:split_point], dates_sorted[split_point:]

print()
print(f'Temporal train set: {len(X_train_temporal):,} games, '
      f'{pd.Timestamp(dates_train.min()).date()} to {pd.Timestamp(dates_train.max()).date()}')
print(f'Temporal test set : {len(X_test_temporal):,} games, '
      f'{pd.Timestamp(dates_test.min()).date()} to {pd.Timestamp(dates_test.max()).date()}')


Unparseable release dates: 0 (0.00%)
Games with a valid date: 20,383 of 20,383

Temporal train set: 16,306 games, 1997-06-30 to 2023-08-18
Temporal test set : 4,077 games, 2023-08-18 to 2025-03-10


## 5. Compare Class Balance: Older vs. Newer Games

Worth reporting regardless of the final Macro F1 result — if reception rates have shifted over
time (e.g. due to platform saturation or changing player expectations), that is itself a
relevant finding for the Discussion.


In [5]:
pct_positive_train = y_train_temporal.mean() * 100
pct_positive_test  = y_test_temporal.mean() * 100

print(f'Class balance — temporal train set (older games): {pct_positive_train:.1f}% Well Received')
print(f'Class balance — temporal test set (newer games)  : {pct_positive_test:.1f}% Well Received')
print(f'Original random-split class balance (both sets)  : ~71.8% Well Received')
print()
if abs(pct_positive_train - pct_positive_test) > 5:
    print(f'Class balance has shifted by {abs(pct_positive_train - pct_positive_test):.1f} '
          f'percentage points between older and newer games — worth mentioning in the Discussion '
          f'regardless of the Macro F1 result below.')
else:
    print('Class balance is broadly stable across time — reception rates have not shifted '
          'substantially between older and newer games in this dataset.')


Class balance — temporal train set (older games): 70.6% Well Received
Class balance — temporal test set (newer games)  : 76.7% Well Received
Original random-split class balance (both sets)  : ~71.8% Well Received

Class balance has shifted by 6.1 percentage points between older and newer games — worth mentioning in the Discussion regardless of the Macro F1 result below.


## 6. Retrain Model D on the Temporal Split

Same hyperparameters and features as the original Model D. Only the train/test partition
changes, from random stratified to chronological.


In [6]:
start = time.time()
model_temporal = CatBoostClassifier(**CB_PARAMS)
model_temporal.fit(X_train_temporal, y_train_temporal)
elapsed = time.time() - start

y_pred_temporal = model_temporal.predict(X_test_temporal)
y_prob_temporal = model_temporal.predict_proba(X_test_temporal)[:, 1]

macro_temporal = f1_score(y_test_temporal, y_pred_temporal, average='macro')
minor_temporal = f1_score(y_test_temporal, y_pred_temporal, pos_label=0)
auc_temporal   = roc_auc_score(y_test_temporal, y_prob_temporal)
acc_temporal   = accuracy_score(y_test_temporal, y_pred_temporal)

print(f'Model D retrained on temporal split ({elapsed:.1f}s):')
print(f'  Macro F1    : {macro_temporal:.4f}')
print(f'  Minority F1 : {minor_temporal:.4f}')
print(f'  AUC-ROC     : {auc_temporal:.4f}')
print(f'  Accuracy    : {acc_temporal:.4f}')


Model D retrained on temporal split (17.8s):
  Macro F1    : 0.6438
  Minority F1 : 0.4650
  AUC-ROC     : 0.7005
  Accuracy    : 0.7336


## 7. Results Comparison: Random Split vs. Temporal Split

In [7]:
print('=== RANDOM SPLIT (NB04) vs. TEMPORAL SPLIT (this notebook) ===')
print(f'{"":<20} {"Macro F1":>10} {"Minority F1":>12} {"AUC":>8} {"Accuracy":>10}')
print('-' * 64)
print(f'{"Random split (NB04)":<20} {0.6690:>10.4f} {0.5172:>12.4f} {0.7394:>8.4f} {0.7385:>10.4f}')
print(f'{"Temporal split (new)":<20} {macro_temporal:>10.4f} {minor_temporal:>12.4f} '
      f'{auc_temporal:>8.4f} {acc_temporal:>10.4f}')
print()
delta = macro_temporal - 0.6690
print(f'Delta (temporal - random): {delta:+.4f} Macro F1')

if abs(delta) < 0.02:
    print('Performance is broadly stable under a temporal split — supports the claim that the ')
    print('system generalises across time, not just across a random shuffle of the same period.')
elif delta < -0.02:
    print('Performance drops meaningfully under a temporal split — worth reporting honestly as ')
    print('evidence that the random-split number may be somewhat optimistic relative to a real ')
    print('deployment scenario where the model only ever sees future games.')
else:
    print('Performance is higher under the temporal split than the random split — report this too, ')
    print('and consider whether the temporal test period happens to contain an easier mix of games.')


=== RANDOM SPLIT (NB04) vs. TEMPORAL SPLIT (this notebook) ===
                       Macro F1  Minority F1      AUC   Accuracy
----------------------------------------------------------------
Random split (NB04)      0.6690       0.5172   0.7394     0.7385
Temporal split (new)     0.6438       0.4650   0.7005     0.7336

Delta (temporal - random): -0.0252 Macro F1
Performance drops meaningfully under a temporal split — worth reporting honestly as 
evidence that the random-split number may be somewhat optimistic relative to a real 
deployment scenario where the model only ever sees future games.


## 8. Summary

This is direct evidence for the "no temporal validation" criticism, rather than a limitation
statement alone. Suggested paper wording once you have the actual numbers: *"To test
generalisation across time, Model D was retrained on the earliest 80% of games by release date
and evaluated on the most recent 20%, rather than a random split. Macro F1 [was stable at /
dropped to / rose to] [X] under this chronological split, compared with 0.6690 under the original
random stratified split, indicating that [the system generalises reasonably across time / some
temporal drift exists that should be monitored in deployment]."*

Whichever way the result lands, this is a stronger response than an unaddressed limitation, it
shows the generalisability question was actually tested, not just acknowledged.

Nothing in NB01–NB15 or in `games_features_t4.csv` was modified by running this notebook.
